# Bank Log-Loss & Cost – Practice Skeleton

**Short name (GitHub):** `Bank_LogLoss_Cost`  
**Domain:** Retail credit / underwriting scorecards  
**Concept source:** Coursera ML C1_W3 Lab04 (logistic loss) + Lab05 (cost J)  
**Language:** Python (NumPy + Matplotlib)

A regional bank has two frozen PD scorecards. You will **not** train weights here (that is `Banking_Logistic_Regression`). You will implement the **loss** and **book-level cost**, then decide which scorecard is more consistent with observed defaults.

Use this notebook to practice. Answers live in **`Bank_LogLoss_Cost_Solution.ipynb`**.

### Learning objectives
- Map $f_{w,b}$ to a **probability of default (PD)** — not a dollar NPL
- Explain why squared error on PD is a bad scorecard objective
- Implement per-loan logistic loss and the book cost $J(w,b)$
- Compare two candidate scorecards the same way Lab05 compared $b=-3$ vs $b=-4$
- Separate **logistic loss** from **expected credit loss** ($\mathrm{PD}\times\mathrm{LGD}\times\mathrm{EAD}$)
- Simulate dirty outcome files and book size; write for Credit Risk / Risk Committee / branch staff

### Data
- `data/bank_dti_1d.csv` — 10 loans, DTI vs default
- `data/bank_scorecard_2d.csv` — 6 applications, utilisation & DTI indices (Lab05 geometry)
- `data/bank_book_practice.csv` — 120-loan practice book


## Inline cheat-sheet

See also **`Bank_LogLoss_Cost_Cheatsheet.docx`**.

| Item | Banking reading | Code / formula |
|------|-----------------|----------------|
| $y=1$ | loan **defaulted** | binary |
| $f$ | predicted **PD** | `sigmoid(w·x+b)` |
| Loss $y=1$ | “we said PD was $f$, it defaulted” | $-\log(f)$ |
| Loss $y=0$ | “we said PD was $f$, it paid” | $-\log(1-f)$ |
| Cost $J$ | mean surprise on the **book** | `mean(L)` |
| Not $J$ | expected $ loss | `PD * LGD * EAD` |
| Clip | never `log(0)` on a sure PD | `[1e-15, 1-1e-15]` |
| Lab05 check | scorecard S1 cheaper than S2 | $J(b=-3)\approx0.367$, $J(b=-4)\approx0.504$ |


## 0. Packages


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("ready")



## 1. Why squared error is the wrong scorecard objective

A PD model outputs $f\in(0,1)$. Squaring $(f-y)$ after a sigmoid is **not** a soup bowl. Credit Risk should not minimise that surface.

### Task 1.1 — load the DTI book


In [ ]:
# TODO: load data/bank_dti_1d.csv → dti (m,), y_def (m,)
# arr = np.loadtxt("data/bank_dti_1d.csv", delimiter=",", skiprows=1)

# YOUR CODE HERE


print("n loans", dti.shape[0], "default rate", y_def.mean())



### Task 1.2 — plot DTI vs outcome


In [ ]:
# TODO: scatter paid vs defaulted against DTI
# YOUR CODE HERE



### Task 1.3 — sigmoid + squared-error cost on PD


In [ ]:
def sigmoid(z):
    """Stable sigmoid for scalars and arrays."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


def squared_pd_cost(x, y, w, b):
    """Mean (PD − y)^2. Do not use this to train."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


print("sigmoid(0) =", sigmoid(0.0))
print("J_sq tight (18, -6.8) =", squared_pd_cost(dti, y_def, 18.0, -6.8))
print("J_sq loose (4, -1.0)  =", squared_pd_cost(dti, y_def, 4.0, -1.0))



## 2. Per-loan logistic loss

$$
L(f,y)=
\begin{cases}
-\log(f) & y=1\ \text{(defaulted)}\\
-\log(1-f) & y=0\ \text{(paid)}
\end{cases}
=
-\big[y\log f+(1-y)\log(1-f)\big]
$$

A scorecard that posts PD = 0.99 on a loan that **paid** is a much worse miss than PD = 0.55 on the same loan.

### Task 2.1 — both forms (clip $f$)


In [ ]:
def loan_loss_piecewise(f, y):
    ### START CODE HERE ###
    
    ### END CODE HERE ###


def loan_loss_compact(f, y):
    ### START CODE HERE ###
    
    ### END CODE HERE ###


for f, y in [(0.20, 1), (0.05, 0), (0.85, 0), (0.90, 1)]:
    print(f"PD={f}, defaulted={int(y)}  piecewise={loan_loss_piecewise(f,y):.4f}  compact={float(loan_loss_compact(np.array([f]), np.array([y]))):.4f}")



### Task 2.2 — draw the two curves vs predicted PD


In [ ]:
# TODO
# YOUR CODE HERE



## 3. Book cost $J(w,b)$

$$
J(w,b)=\frac{1}{m}\sum_{i=1}^{m} L\big(f_{w,b}(x^{(i)}), y^{(i)}\big)
$$

This is **not** the bank’s credit-loss number. It is how surprised the scorecard is, on average.

### Task 3.1 — loop (original lab style)
Handle 1-D `x` and 2-D `X`.


In [ ]:
def compute_cost_logistic(X, y, w, b):
    ### START CODE HERE ###
    
    ### END CODE HERE ###


print("1-D J tight =", compute_cost_logistic(dti, y_def, 18.0, -6.8))
print("1-D J loose =", compute_cost_logistic(dti, y_def, 4.0, -1.0))



### Task 3.2 — vectorized alternate


In [ ]:
def compute_cost_logistic_vec(X, y, w, b):
    ### START CODE HERE ###
    
    ### END CODE HERE ###


print("vec matches loop?",
      np.isclose(compute_cost_logistic(dti, y_def, 18.0, -6.8),
                 compute_cost_logistic_vec(dti, y_def, 18.0, -6.8)))



## 4. Two scorecards on six applications (Lab05 geometry)

`util_idx` and `dti_idx` are internal risk indices. Compare
- **S1:** $w=(1,1),\; b=-3$  →  cut `util + dti = 3`
- **S2:** $w=(1,1),\; b=-4$  →  cut `util + dti = 4`

Expected: $J_{S1}\approx 0.3669$, $J_{S2}\approx 0.5037$.


In [ ]:
# TODO: load data/bank_scorecard_2d.csv
# columns: util_idx, dti_idx, defaulted, credit_score, dti

# YOUR CODE HERE


w = np.array([1.0, 1.0])
print("J S1 b=-3 =", compute_cost_logistic(X2, y2, w, -3))
print("J S2 b=-4 =", compute_cost_logistic(X2, y2, w, -4))



### Task 4.1 — plot both cuts and label paid vs defaulted


In [ ]:
# TODO
# YOUR CODE HERE



## 5. More practice

### 5.1 Three files
Compute loss and mean $J$.

| Loan | PD | Defaulted |
|------|-----|-----------|
| L1 | 0.12 | 1 |
| L2 | 0.08 | 0 |
| L3 | 0.78 | 0 |


In [ ]:
# TODO
# YOUR CODE HERE



### 5.2 Practice book
Load `data/bank_book_practice.csv`. Score two crude scorecards on **centered** features
`x0 = credit_score - 660`, `x1 = dti - 0.32`:

- Book model: $w=(-0.012,\ 8.5),\; b=0$
- Naive 50/50: $w=(0,0),\; b=0$  (should be $\approx 0.693$)


In [ ]:
# TODO
# YOUR CODE HERE



### 5.3 Logistic loss vs expected credit loss
For one defaulted loan with EAD = 20_000 and LGD = 0.45, compare
- logistic loss at PD = 0.10 vs PD = 0.80
- dollar ECL = PD × LGD × EAD at the same two PDs

They do not move on the same scale. Say in one sentence when you would quote each number.


In [ ]:
# TODO
# YOUR CODE HERE



## 6. Simulation — dirty files and book size

Edit the knobs. Frozen scorecard = the “book model” from 5.2.
You should see $J$ settle as $m$ grows and rise as outcome-misfile rate rises.


In [ ]:
# ===== knobs =====
TRUE_W = np.array([-0.012, 8.5])
TRUE_B = 0.0
M_LIST = [40, 80, 160, 320, 640]
MISFILE_LIST = [0.00, 0.04, 0.08, 0.15, 0.25]
N_REPS = 18
SEED = 11
# =================

def make_book(m, misfile, seed):
    rng = np.random.default_rng(seed)
    cs = rng.normal(690, 55, m).clip(520, 820)
    dti_s = rng.normal(0.34, 0.11, m).clip(0.08, 0.75)
    X = np.column_stack([cs - 660.0, dti_s - 0.32])
    p = sigmoid(X @ TRUE_W + TRUE_B)
    y = (rng.random(m) < p).astype(float)
    nflip = int(misfile * m)
    if nflip:
        idx = rng.choice(m, size=nflip, replace=False)
        y[idx] = 1 - y[idx]
    return X, y

# TODO: error-bar plot of mean J vs m (misfile=0.04)
# TODO: error-bar plot of mean J vs misfile (m=200)
# YOUR CODE HERE



## 7. Audience notes (3–5 sentences each)

1. **Credit Risk / model validation** — $J$, S1 vs S2, clip policy, why not MSE.  
2. **Risk Committee / executive** — which cut is cheaper on surprise, what you need before rollout. No formulas.  
3. **Branch / non-specialist** — DTI story only; “how shocked the rule is when a loan pays or breaks”.


In [ ]:
credit_risk_note = """YOUR TEXT"""
committee_note = """YOUR TEXT"""
branch_note = """YOUR TEXT"""
print(credit_risk_note); print(committee_note); print(branch_note)



## 8. Flowchart


In [ ]:
from IPython.display import Image, display
import os
for p in ["bank_logloss_cost_flowchart.png", "bank_logloss_curves.png",
          "bank_logloss_heatmap.png", "bank_logloss_boundaries.png"]:
    print(p, "OK" if os.path.exists(p) else "missing")
    if os.path.exists(p):
        display(Image(p, width=620))



## Checklist
- [ ] Sigmoid stable
- [ ] Piecewise loss = compact loss
- [ ] Loop $J$ = vectorized $J$
- [ ] $J_{S1} < J_{S2}$
- [ ] ECL sentence written
- [ ] Simulation knobs move the charts
- [ ] Three audience paragraphs
